# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and name
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}\n")

# For demonstration, select the first record set (if exists)
if len(record_sets):
    chosen_recordset = record_sets[0]
    print(f"Fields in RecordSet '{chosen_recordset.name}' (@id: {chosen_recordset.id}):")
    for field in chosen_recordset.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
  
for record_set_id in record_set_ids:
    # Retrieve records for the record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} rows from RecordSet {record_set_id}.")
    if len(dataframes[record_set_id].columns) > 0:
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")

# Select a record set for further EDA (default to the first one if available)
if len(record_set_ids):
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None

if selected_record_set_id:
    print(f"\nHead of dataframe for {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automatically pick a numeric field from the DataFrame
import numpy as np

df = dataframes[selected_record_set_id]

# Infer a numeric field to use (fallback to first float/integer column)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_columns):
    numeric_field = numeric_columns[0]
    print(f"Using numeric field: {numeric_field}")
else:
    print("No numeric columns found in the data. Please review the data and pick an appropriate field.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records.")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (pick the first non-numeric column)
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    group_field = cat_cols[0] if len(cat_cols) else None
    if group_field:
        print(f"\nGrouping by categorical field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())
    else:
        print("No categorical grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    plt.show()

    # If a group field exists, show boxplots
    if group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was successfully loaded with `mlcroissant` using its Croissant schema.
- Key record sets and fields were identified and loaded into dataframes.
- Basic exploratory analysis and visualization were performed. To extend this analysis, integrate with domain-specific analytics or focus on specific biological questions relevant to the mass spectrometry data.